# Blazar Population ML — Project 1

**Goal:** Apply classical ML to the 4LAC-DR3 blazar catalog to recover constraints on jet baryon loading ($\mu$) from multi-wavelength observational features, and to characterize the blazar population via unsupervised clustering.

**Reference:** Rueda-Becerril, Harrison & Giannios (2021, MNRAS 501, 4092)

**Catalogs:**
- 4LAC-DR3: Ajello et al. (2022, arXiv:2209.12070), high-latitude, $|b| > 10^\circ$, `table-4LAC-DR3-h.fits`
- MOJAVE-XVII: apparent velocities from VLBI monitoring, `VizieR-MOJAVE-XVII.fit`

---

## Analysis Structure

- **Part 1 (full catalog, ~3,400 sources):** catalog-native features only, no kinematics
- **Part 2 (MOJAVE cross-matched, ~334 sources):** adds $\beta_{\text{app}}$ and $\Gamma_{\text{min}}$

**Notebook sections:**
1. Setup and imports
2. Load and inspect 4LAC-DR3
3. Load and inspect MOJAVE-XVII
4. Cross-match: 4LAC-DR3 × MOJAVE-XVII
5. Feature derivation and unified dataframes
6. Missingness audit
7. Exploratory data analysis (EDA)
8. Dimensionality reduction: PCA and UMAP
9. Clustering: GMM and HDBSCAN

## 1. Setup and Imports

In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path

from astropy.io import fits
from astropy.table import Table, vstack
from astropy.coordinates import SkyCoord
from astropy.cosmology import FlatLambdaCDM
import astropy.units as u

from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
from sklearn.mixture import GaussianMixture

import warnings
warnings.filterwarnings('ignore')

# Cosmology (consistent with 2021 paper)
cosmo = FlatLambdaCDM(H0=70, Om0=0.3)

# Paths
DATA_DIR = Path('data/')
LAC_H  = DATA_DIR / 'table-4LAC-DR3-h.fits'
MOJAVE = DATA_DIR / 'VizieR-MOJAVE-XVII.fit'

# Plot style
plt.rcParams.update({'figure.dpi': 120, 'font.size': 11})
CLASS_COLORS = {'bll': 'steelblue', 'fsrq': 'tomato', 'bcu': 'gray'}
SED_COLORS   = {'lsp': 'tomato', 'isp': 'goldenrod', 'hsp': 'steelblue'}

## 2. Load and Inspect 4LAC-DR3

Use `table-4LAC-DR3-h.fits` (high Galactic latitude, $|b| > 10^\circ$) as the primary catalog.
`table-4LAC-DR3-l.fits` covers low-latitude sources and is excluded from the main analysis.

**Tasks:**
- [X] Print header data unit (HDU) structure and column list with units
- [X] Check row count: expect 3,407 sources
- [X] Print CLASS and SED_class distributions (note mixed-case entries: normalize to lowercase)
- [X] Check for masked values vs. sentinel zeros, especially in `Redshift`, `nu_syn`, `nuFnu_syn`

### Inspecting `table-4LAC-DR3-h.fits`

In [2]:
def safe_float(col):
    return np.ma.filled(col.data, np.nan).astype(float)

def safe_str(col):
    return np.array([str(v).strip() if v is not np.ma.masked else '' for v in col])

def safe_str_lower(col):
    return np.array([str(v).strip().lower() if v is not np.ma.masked else '' for v in col])

In [3]:
# Exploring header data unit (HDU)
with fits.open(LAC_H) as hdul:
    for i, hdu in enumerate(hdul):
        print(f"HDU {i}: {type(hdu).__name__}")
        if hasattr(hdu, 'columns') and hdu.columns:
            print(f"Rows: {len(hdu.data)}")
            print(f"Columns ({len(hdu.columns)}):")
            for col in hdu.columns:
                print(f"  {col.name:30s}  {col.format:6s}  {col.unit if col.unit else ''}")

HDU 0: PrimaryHDU
HDU 1: BinTableHDU
Rows: 3407
Columns (41):
  Source_Name                     18A     
  DataRelease                     I       
  RAJ2000                         E       deg
  DEJ2000                         E       deg
  GLON                            E       deg
  GLAT                            E       deg
  Signif_Avg                      E       
  Flux1000                        E       ph cm-2 s-1
  Unc_Flux1000                    E       ph cm-2 s-1
  Energy_Flux100                  E       erg cm-2 s-1
  Unc_Energy_Flux100              E       erg cm-2 s-1
  SpectrumType                    18A     
  PL_Index                        E       
  Unc_PL_Index                    E       
  Pivot_Energy                    E       MeV
  LP_Index                        E       
  Unc_LP_Index                    E       
  LP_beta                         E       
  Unc_LP_beta                     E       
  Flags                           I       
  CLASS          

In [4]:
lac = Table.read(LAC_H)
lac_len = len(lac)

cls_mixed = safe_str(lac['CLASS'])
cls_lower = safe_str_lower(lac['CLASS'])
sed_arr = safe_str_lower(lac['SED_class'])

# Finding unique values in CLASS column
classes, cls_counts = np.unique(cls_lower, return_counts=True)
classes_mixed, counts_mixed = np.unique(cls_mixed, return_counts=True)

# Finding unique values in SED_class column
sed, sed_counts = np.unique(sed_arr, return_counts=True)


### `CLASS` distribution

In [5]:
for cl, n in sorted(zip(classes_mixed, counts_mixed), key=lambda x: -x[1]):
    print(f"{cl:10s}  {n:5d}  ({100 * n / lac_len:.1f}%)")

bll          1357  (39.8%)
bcu          1208  (35.5%)
fsrq          715  (21.0%)
FSRQ           40  (1.2%)
rdg            36  (1.1%)
BLL            22  (0.6%)
RDG             6  (0.2%)
agn             6  (0.2%)
css             5  (0.1%)
NLSY1           4  (0.1%)
nlsy1           4  (0.1%)
ssrq            2  (0.1%)
AGN             1  (0.0%)
sey             1  (0.0%)


### `SED_class` distribution

In [6]:
for s, n in sorted(zip(sed, sed_counts), key=lambda x: -x[1]):
    label = s if s else '(missing)'
    print(f"{label:10s}  {n:5d}  ({100 * n / lac_len:.1f}%)")

lsp          1564  (45.9%)
(missing)     777  (22.8%)
hsp           551  (16.2%)
isp           515  (15.1%)


### Missingness for key features

In [7]:
key_cols = [
    'PL_Index', 'LP_Index', 'Redshift', 'nu_syn', 'nuFnu_syn', 'HE_EPeak',
    'HE_nuFnuPeak', 'Variability_Index', 'Frac_Variability', 'Energy_Flux100'
]

for col in key_cols:
    arr = safe_float(lac[col])
    n_nan = np.sum(np.isnan(arr))
    n_zero = np.sum(arr == 0)
    print(f"{col:25s}  missing: {n_nan:4d} ({100 * n_nan / lac_len:5.1f}%)  zero: {n_zero:4d}")
    
print(f"\nN = {lac_len}")

PL_Index                   missing:    0 (  0.0%)  zero:    0
LP_Index                   missing:    0 (  0.0%)  zero:    0
Redshift                   missing:    0 (  0.0%)  zero:    0
nu_syn                     missing:    0 (  0.0%)  zero:  777
nuFnu_syn                  missing:    0 (  0.0%)  zero:  777
HE_EPeak                   missing:  518 ( 15.2%)  zero:    0
HE_nuFnuPeak               missing:    0 (  0.0%)  zero:    0
Variability_Index          missing:    0 (  0.0%)  zero:    0
Frac_Variability           missing:    0 (  0.0%)  zero:  874
Energy_Flux100             missing:    0 (  0.0%)  zero:    0

N = 3407


### Derived features feasibility

In [8]:
he = safe_float(lac['HE_nuFnuPeak'])
syn = safe_float(lac['nuFnu_syn'])
flux = safe_float(lac['Energy_Flux100'])
z = np.nan_to_num(lac['Redshift'].data, nan=0.0, posinf=0.0, neginf=0.0)

has_sed = (syn > 0) & (he > 0) & (~np.isnan(he))
has_z = z > 0
has_lum = has_z & (flux > 0) & (~np.isnan(flux))
has_cd = has_sed
is_blazar = np.isin(cls_lower, ['bll', 'fsrq'])
is_bcu = cls_lower == 'bcu'

### Compton Dominand and Gamma-ray Luminosity

In [9]:
cd_valid = np.sum(has_cd)
lum_valid = np.sum(has_lum)

print(f"Compton dominance computable:    {cd_valid:4d}/{lac_len}  ({100 * cd_valid / lac_len:.1f}%)")
print(f"Gamma-ray luminosity computable: {lum_valid:4d}/{lac_len}  ({100 * lum_valid / lac_len:.1f}%)")

Compton dominance computable:    2247/3407  (66.0%)
Gamma-ray luminosity computable: 1806/3407  (53.0%)


### Redshift

Redshift 0 means "no measurement", not actually z = 0.

In [10]:
print(f"z > 0 (measured):    {np.sum(z > 0):4d} ({100 * np.sum(z > 0) / lac_len:.1f}%)")
print(f"z = 0 (no redshift): {np.sum(z == 0):4d} ({100 * np.sum(z == 0) / lac_len:.1f}%)")

z > 0 (measured):    1806 (53.0%)
z = 0 (no redshift): 1601 (47.0%)


### `CLASS` case normalization issue

Mixed case entries detected: bll vs BLL, fsrq vs FSRQ; need normalization

In [11]:
for cl, n in sorted(zip(classes, cls_counts), key=lambda x: -x[1]):
    print(f"{cl:5s} + {cl.upper():5s} = {n:5d} ({100 * n / cls_counts.sum():.1f}%)")

bll   + BLL   =  1379 (40.5%)
bcu   + BCU   =  1208 (35.5%)
fsrq  + FSRQ  =   755 (22.2%)
rdg   + RDG   =    42 (1.2%)
nlsy1 + NLSY1 =     8 (0.2%)
agn   + AGN   =     7 (0.2%)
css   + CSS   =     5 (0.1%)
ssrq  + SSRQ  =     2 (0.1%)
sey   + SEY   =     1 (0.0%)


### BCU SED class breakdown

In [12]:
bcu_sed, bcu_counts = np.unique(sed_arr[is_bcu], return_counts=True)

for s, n in sorted(zip(bcu_sed, bcu_counts), key=lambda x: -x[1]):
    label = s if s else '(no SED class)'
    print(f"{label:15s}  {n:4d}")

lsp               508
(no SED class)    448
isp               135
hsp               117


### Confirmed blazars (BL Lacs + FSRQs) by SED class

In [13]:
bl_sed, bl_counts = np.unique(sed_arr[is_blazar], return_counts=True)

for s, n in sorted(zip(bl_sed, bl_counts), key=lambda x: -x[1]):
    label = s if s else '(no SED class)'
    print(f"{label:15s}  {n:4d}")

lsp              1025
hsp               429
isp               367
(no SED class)    313


### Summary

**Effective Sample Sizes**

In [14]:
print(f"Total sources:                     {lac_len:4d}")
print(f"Confirmed blazars (BL Lac + FSRQ): {np.sum(is_blazar):4d}  ({100 * np.sum(is_blazar) / lac_len:.1f}%)")
print(f"BCU:                               {np.sum(is_bcu):4d}  ({100 * np.sum(is_bcu) / lac_len:.1f}%)")
print(f"With SED peak (nu_syn > 0):        {np.sum(syn > 0):4d}  ({100 * np.sum(syn > 0) / lac_len:.1f}%)")
print(f"With Compton dominance:            {np.sum(has_cd):4d}  ({100 * np.sum(has_cd) / lac_len:.1f}%)")
print(f"With redshift:                     {np.sum(has_z):4d}  ({100 * np.sum(has_z) / lac_len:.1f}%)")
print(f"With luminosity (z * flux):        {np.sum(has_lum):4d}  ({100 * np.sum(has_lum) / lac_len:.1f}%)")
print(f"Confirmed blazars + all above:     {np.sum(is_blazar & has_cd & has_lum):4d}  ({100 * np.sum(is_blazar & has_cd & has_lum) / lac_len:.1f}%)")

Total sources:                     3407
Confirmed blazars (BL Lac + FSRQ): 2134  (62.6%)
BCU:                               1208  (35.5%)
With SED peak (nu_syn > 0):        2630  (77.2%)
With Compton dominance:            2247  (66.0%)
With redshift:                     1806  (53.0%)
With luminosity (z * flux):        1806  (53.0%)
Confirmed blazars + all above:     1203  (35.3%)


**1. `CLASS` distribution (after normalization):**
- BL Lac: 1,379 (40.5%)
- BCU: 1,208 (35.5%) -- largest uncertainty population
- FSRQ: 755 (22.2%)
- Radio Galaxy: 42 (1.2%)
- Other (NLSY1, AGN, CSS, SSRQ, SEY): 23 (~0.6%)

**2. `SED_class` distribution:**
- Shows Low synchrotron peaked (LSP) sources dominate at 45.9%, while 777 entries (22.8%) missing SED classification.
- The missing SED class correspond exactly to sources lacking synchrotron peak measurements (i.e., zero `nu_syn` or `nuFnu_syn` values), predominantly BCUs.
- High synchrotron peaked (HSP) and Intermediate synchrotron peaked (ISP) make up the remaining 31.3% of classified sources.

**3.Missingness:**
- Looking at the missing patterns, `PL_index` and `LP_index` are fully populated, but there are gaps elsewhere that need investigation.
- Redshift is encoded as 0 for unmeasured sources rather than NaN.
- `HE_EPeak` has 518 missing entries (15.2%), while `Energy_Flux100` is complete.
- For derived features, Compton dominance has good coverage at 66%, but gamma-ray luminosity is constrained by redshift availability at only 53%.

Looking at the effective sample sizes, the full catalog has 3,407 sources, but only 1,806 have redshift data needed for luminosity calculations, and just 1,208 are BCUs (the actual classification targets).

The confirmed blazars (BLL + FSRQ) total 2,134 sources, which gives a solid training set.

---

## 3. Load and Inspect MOJAVE-XVII

The file has 6 HDUs. The relevant ones are:
- **HDU 1 (204 sources):** BL Lac-dominated. Columns include `ID`, `_RA`, `_DE`, `betaMax`, `e_betaMax`, `Cl`
- **HDU 2 (230 sources):** FSRQ/Quasar-dominated. Same key columns, also has `Smax` (peak flux)
- **HDU 4 (1,743 rows):** per-component kinematics — one row per tracked jet component, not used here

HDU 1 and HDU 2 have **zero overlap** in source IDs. Combine and deduplicate to get 434 unique sources.

**Warning:** The `SimbadName` column in HDUs 1 and 2 has a malformed FITS format that will crash `Table(hdul[i].data)`. Extract only the columns you need by iterating over `hdul[i].data` rows directly.

**Tasks:**
- [ ] Load HDU 1 and HDU 2, extract: `ID`, `_RA`, `_DE`, `betaMax`, `e_betaMax`, `Cl`
- [ ] Confirm zero overlap between HDU 1 and HDU 2 source IDs
- [ ] Combine and deduplicate on `ID`
- [ ] Report `betaMax` distribution; how many sources have betaMax = 0?

In [15]:
with fits.open(MOJAVE) as hdul:
    for i, hdu in enumerate(hdul):
        print(f"HDU {i}: {type(hdu).__name__}")
        if hasattr(hdu, 'columns') and hdu.columns:
            print(f"Rows: {len(hdu.data)}")
            print(f"Columns ({len(hdu.columns)}):")
            for col in hdu.columns:
                print(f"  {col.name:30s}  {col.format:6s}  {col.unit if col.unit else ''}")

HDU 0: PrimaryHDU
HDU 1: TableHDU
Rows: 204
Columns (22):
  Nf                              I3      
  ID                              A8      
  Nacc                            I3      
  Nvm                             I3      
  f_ID                            A3      
  OName                           A20     
  Cl                              A1      
  z                               F7.4    
  lognu                           F4.1    log(Hz)
  r_lognu                         A2      
  muMax                           F6.1    uarcsec/yr
  e_muMax                         F5.1    uarcsec/yr
  f_muMax                         A1      
  betaMax                         F7.4    
  e_betaMax                       F7.4    
  Ref                             A19     
  L11                             I3      
  L13                             I3      
  SimbadName                      F       
  NED                             A3      
  _RA                             F9.5    deg
  _DE    

In [ ]:
def load_mojave_hdu(hdul, i):
    d = hdul[i].data
    return Table({
        'ID': np.array(safe_str(d['ID'])),
        'RA': np.array(safe_float(d['_RA'])),
        'DEC': np.array(safe_float(d['_DE'])),
        'betaMax': np.array(safe_float(d['betaMax'])),
        'e_betaMax': np.array(safe_float(d['e_betaMax']))
        'Cl': np.array(safe_float(d['e_betaMax']))
    })

In [17]:
with fits.open(MOJAVE) as hdul:
    moj = vstack([load_mojave_hdu(hdul, 1), load_mojave_hdu(hdul, 2)])
_, fi = np.unique(moj['ID'], return_index=True)
moj = moj[np.sort(fi)]
moj

ID,RA,DEC,betaMax,e_betaMax
str8,float64,float64,float64,float64
0003+380,1.48823,38.33754,4.61,0.36
0006+061,2.26638,6.47257,0.0,0.0
0011+189,3.4849,19.17831,4.54,0.46
0010+405,3.37971,40.86032,6.92,0.64
0015-054,4.39924,-5.2116,0.72,0.28
0019+058,5.63517,6.13452,0.0,0.0
0027+056,7.44123,5.91131,1.45,0.38
0026+346,7.30934,34.94229,1.76,0.7
0035+413,9.60351,41.61833,7.4,0.31


## 4. Cross-Match: 4LAC-DR3 × MOJAVE-XVII

Use `SkyCoord.match_to_catalog_sky` for positional matching.

- **4LAC-DR3 coords:** `RA_Counterpart` / `DEC_Counterpart` (counterpart position, more precise than LAT position). Filter out sources with masked counterpart coords before building the SkyCoord.
- **MOJAVE coords:** `_RA`, `_DE` (J2000, degrees)

Test tolerances 1", 2", 3", 5". If match count is stable across this range, 2" is safe — VLBI positions are precise enough that all real matches will be well under 1".

**Tasks:**
- Run cross-match, report match count and separation statistics
- Inspect CLASS and SED_class breakdown of matched sources
- Note the selection bias: MOJAVE selects radio-bright sources, mostly LSP/FSRQ

### Loading 4LAC-DR3

In [18]:
def lac_features(lac_sub):
    z = safe_float(lac_sub['Redshift']); z = np.where(z <= 0, np.nan, z)
    nu = safe_float(lac_sub['nu_syn']); nu = np.where(nu <= 0, np.nan, nu)
    nf = safe_float(lac_sub['nuFnu_syn']); nf = np.where(nf <= 0, np.nan, nf)
    hp = safe_float(lac_sub['HE_EPeak'])
    hn = safe_float(lac_sub['HE_nuFnuPeak']); hn = np.where(hn <=0, np.nan, hn)
    he_erg = (hn * u.MeV / u.cm**2 / u.s).to(u.erg / u.cm**2 / u.s).value
    cd = np.where((~np.isnan(he_erg)) & (~np.isnan(nf)) & (nf > 0), he_erg / nf, np.nan)
    ef = safe_float(lac_sub['Energy_Flux100'])
    vi = safe_float(lac_sub['Variability_Index'])
    fv = safe_float(lac_sub['Frac_Variability']); fv = np.where(fv <= 0, np.nan, fv)
    
    with warnings.catch_warnings():
        warnings.simplefilter('ignore')
    
    # Calculating luminosity distance
    dl = np.where(
        ~np.isnan(z),
        cosmo.luminosity_distance(np.where(~np.isnan(z), z, 0.1)).to(u.cm).value,
        np.nan
    )
    lum = np.where(~np.isnan(z), 4. * np.pi * dl**2 * ef, np.nan)

    return dict(
        class_=safe_str(lac_sub['CLASS']),
        sed_class=safe_str(lac_sub['SED_class']),
        redshift=z,
        pl_index=safe_float(lac_sub['PL_Index']),
        lp_alpha=safe_float(lac_sub['LP_Index']),
        lp_beta=safe_float(lac_sub['LP_beta']),
        log_nu_syn=np.log10(np.where(nu > 0, nu, np.nan)),
        log_nuFnu_syn=np.log10(np.where(nf > 0, nf, np.nan)),
        log_he_epeak=np.log10(np.where(hp > 0, hp, np.nan)),
        log_compton_dom=np.log10(np.where(cd > 0, cd, np.nan)),
        log_gamma_lum=np.log10(np.where(lum > 0, lum, np.nan)),
        var_index=vi,
        frac_var=fv,
    )

In [19]:
lac_feat = lac_features(lac)
df1 = pd.DataFrame(
    {
        'source_name': [str(v).strip() for v in lac['Source_Name']],
        'ra': safe_float(lac['RAJ2000']),
        'dec': safe_float(lac['DEJ2000']),
        'class': lac_feat['class_'],
        'sed_class': lac_feat['sed_class'],
        **{k: v for k, v in lac_feat.items() if k not in ['class_', 'sed_class']}
    }
)
df1.head(10)

,source_name,ra,dec,class,sed_class,redshift,pl_index,lp_alpha,lp_beta,log_nu_syn,log_nuFnu_syn,log_he_epeak,log_compton_dom,log_gamma_lum,var_index,frac_var
0,4FGL J0001.2+4741,0.3126,47.685902,bcu,ISP,NaN,2.271696,2.254081,0.012156,14.0000,-12.429871,NaN,NaN,NaN,25.313953,0.675882
1,4FGL J0001.2-0747,0.3151,-7.797100,bll,LSP,NaN,2.116692,2.078927,0.051182,13.9600,-11.712922,2.901076,-0.143536,NaN,46.780693,0.406565
2,4FGL J0001.4-0010,0.3717,-0.169900,bll,LSP,0.461516,1.939160,1.661223,0.132438,12.5575,-12.606731,4.189645,-0.013938,44.989548,9.272764,NaN
3,4FGL J0001.5+2113,0.3815,21.218300,fsrq,ISP,1.106000,2.654060,2.514159,0.159319,14.2000,-11.938370,1.817433,1.089770,47.233789,1910.935791,0.996138
4,4FGL J0001.6-4156,0.4165,-41.942501,bcu,HSP,NaN,1.775175,1.693865,0.072754,15.8650,-11.720573,4.520856,-0.391878,NaN,26.393343,0.490977
5,4FGL J0001.8-2153,0.4647,-21.886499,bcu,LSP,NaN,1.876663,1.716620,0.405808,13.2200,-12.012512,3.798033,-0.593747,NaN,24.557972,0.902851
6,4FGL J0002.1-6728,0.5378,-67.474602,bll,,NaN,1.847773,1.593436,0.186790,NaN,NaN,3.969671,NaN,NaN,12.530725,0.174254
7,4FGL J0002.3-0815,0.5937,-8.265200,bcu,LSP,NaN,2.092075,2.060727,0.146869,13.8800,-12.398651,3.441642,-0.230340,NaN,13.014215,0.096862
8,4FGL J0002.4-5156,0.6131,-51.935501,bcu,,NaN,1.914456,1.535265,0.368987,NaN,NaN,3.883515,NaN,NaN,17.686310,0.571694
9,4FGL J0003.1-5248,0.7817,-52.807098,bcu,,NaN,1.915512,1.859399,0.047373,NaN,NaN,4.175021,NaN,NaN,7.998425,NaN


In [20]:
# Cross-match 4LAC-DR3 x MOJAVE-XVII


## 5. Feature Derivation and Unified Dataframes

Build two clean pandas DataFrames.

**Part 1 — all 3,407 sources, catalog-native + derived features:**

| Feature | Source column | Notes |
|---------|--------------|-------|
| `class` | `CLASS` | Lowercase, normalize |
| `sed_class` | `SED_class` | Lowercase |
| `redshift` | `Redshift` | Treat 0 as NaN |
| `pl_index` | `PL_Index` | Photon spectral index |
| `lp_alpha` | `LP_Index` | Log-parabola α |
| `lp_beta` | `LP_beta` | Log-parabola β (curvature) |
| `log_nu_syn` | log10(`nu_syn`) | Treat 0 as NaN |
| `log_nuFnu_syn` | log10(`nuFnu_syn`) | Treat 0 as NaN |
| `log_he_epeak` | log10(`HE_EPeak`) | MeV, treat 0 as NaN |
| `log_compton_dom` | log10(`HE_nuFnuPeak` × 1.602e-6 / `nuFnu_syn`) | Convert MeV/cm²/s → erg/cm²/s |
| `log_gamma_lum` | log10(4π d_L² × `Energy_Flux100`) | d_L from cosmo.luminosity_distance(z) |
| `var_index` | `Variability_Index` | |
| `frac_var` | `Frac_Variability` | Treat ≤ 0 as NaN |

**Part 2 — 334 MOJAVE-matched sources, adds:**

| Feature | Notes |
|---------|-------|
| `beta_app` | `betaMax` from MOJAVE, in units of c |
| `e_beta_app` | `e_betaMax` |
| `gamma_min` | sqrt(1 + β_app²) — lower bound on Lorentz factor; flag betaMax=0 sources |

Save both as parquet.

### Build Part 1 dataframe (all 4LAC-DR3-h sources)

In [21]:
# Build Part 2 dataframe (MOJAVE cross-matched sources)


In [22]:
# Save both dataframes


## 6. Missingness Audit

Document missingness for every feature. This drives feature set decisions in all downstream steps.

**Tasks:**
- Print % missing per feature for Part 1 and Part 2
- Plot a missingness heatmap (a sample of 500 rows is sufficient)
- Identify co-missing features: `nu_syn`=0 and `SED_class`=missing are the same 777 sources
- **Decision:** for Part 1 clustering, run with two feature sets: (a) SED features only (no luminosity), (b) SED + luminosity (redshift-complete subsample only). Compare results.

In [23]:
# Missingness audit — Part 1


In [24]:
# Missingness audit — Part 2


## 7. Exploratory Data Analysis

Understand the structure of the data before applying any ML. Any surprise here should change the modeling approach.

### 7.1 Univariate distributions
Histograms / KDE per feature, colored by CLASS (BLL=blue, FSRQ=red, BCU=gray). Look for bimodality, skewness, outliers.

### 7.2 BLL vs. FSRQ separation
Pairplot: `log_nu_syn`, `log_compton_dom`, `redshift`. The blazar sequence predicts BLLs at higher synchrotron peak, lower Compton dominance, and generally lower redshift than FSRQs.

### 7.3 Blazar sequence
Scatter: `log_nu_syn` vs. `log_compton_dom`, colored by SED_class (LSP/ISP/HSP). This is the canonical blazar sequence diagram — does the data reproduce it?

### 7.4 Correlation matrix
Pearson correlation heatmap on the complete-case subset. Flag strongly correlated pairs for the modeling phase.

In [25]:
# 7.1 Univariate distributions


In [26]:
# 7.2 BLL vs FSRQ pairplot


In [27]:
# 7.3 Blazar sequence: log_nu_syn vs log_compton_dom


In [28]:
# 7.4 Correlation matrix


## 8. Dimensionality Reduction: PCA and UMAP

**Feature set:** complete-case sources on core SED features: `pl_index`, `log_nu_syn`, `log_nuFnu_syn`, `log_compton_dom`, `var_index`. StandardScaler before both methods.

### 8.1 PCA
- Scree plot (explained variance ratio)
- PC1 vs PC2 colored by: CLASS, SED_class, log_nu_syn
- Loadings: does PC1 correspond to the blazar sequence?

### 8.2 UMAP
- `n_neighbors=15`, `min_dist=0.1`, `random_state=42`
- Plot colored by CLASS and SED_class
- Key question: where do BCU sources land relative to confirmed BLL and FSRQ?
- Install if needed: `pip install umap-learn`

In [29]:
# 8.1 PCA


In [30]:
# 8.2 UMAP


## 9. Clustering: GMM and HDBSCAN

**Goal:** let the data reveal its own groupings without imposing BLL/FSRQ labels. Then compare to known classifications.

### 9.1 Gaussian Mixture Model
- Fit for k = 2, 3, 4, 5 on the same feature set as Section 8
- Select k using BIC and AIC
- Plot cluster assignments on the UMAP embedding
- Confusion matrix: GMM clusters vs. BLL/FSRQ/BCU labels

### 9.2 HDBSCAN
- Fit on UMAP embedding (2D), not raw features
- Start with `min_cluster_size=30`
- Inspect noise points (label = -1): are they predominantly BCU?
- Install if needed: `pip install hdbscan`

### 9.3 BCU probabilistic classification
- Using the GMM posterior probabilities, assign soft class labels to BCU sources
- Report: how many BCUs fall clearly in one class? How many are genuinely intermediate?
- This answers the secondary scientific question

In [31]:
# 9.1 GMM


In [32]:
# 9.2 HDBSCAN


In [33]:
# 9.3 BCU probabilistic classification
